# Problem 6 (100 points)

Convolutional neural networks exploit spatial structure by sliding learned filters across an image. In this problem, you will implement the 2D convolution and max-pooling operations **from scratch**, derive output-shape formulas, and assemble a working CNN for image classification—all without using `nn.Conv2d` or `nn.MaxPool2d`.

We use the following notation in this problem.
- $x \in \mathbb{R}^{B \times C_{\text{in}} \times H \times W}$ — input tensor (batch, channels, height, width).
- $w \in \mathbb{R}^{C_{\text{out}} \times C_{\text{in}} \times K_h \times K_w}$ — convolution kernel.
- $P$ — zero-padding added to each side.
- $S$ — stride.
- Output spatial size: $H_{\text{out}} = \lfloor (H - K + 2P) / S \rfloor + 1$.

In [ ]:
# Run code in this cell

"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(42)

> WARNING !!!
>
- Beyond importing libraries/modules/classes/functions in the preceding cell, you are **NOT allowed to import anything else for the following purposes**:
    - **As a part of your final solution.**
    - **Temporarily import something to assist you to get a solution.**
- Do **NOT** use `nn.Conv2d`, `nn.MaxPool2d`, or `F.conv2d` in Parts 1–4. You may use `F.unfold` and `F.pad`.

## Part 1 (10 points, coding task)

**Do the following tasks.**

Implement `conv2d_output_shape(H_in, W_in, K, P, S)` that returns the tuple `(H_out, W_out)`.

$$H_{\text{out}} = \left\lfloor\frac{H_{\text{in}} - K + 2P}{S}\right\rfloor + 1$$

The same formula applies to width.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def conv2d_output_shape(H_in, W_in, K, P=0, S=1):
    """Return (H_out, W_out) for a Conv2d layer."""
    ...

""" END OF THIS PART """

In [ ]:
""" VERIFICATION """
assert conv2d_output_shape(32, 32, 3, P=1, S=1) == (32, 32),   "same padding"
assert conv2d_output_shape(32, 32, 3, P=0, S=1) == (30, 30),   "valid padding"
assert conv2d_output_shape(32, 32, 3, P=1, S=2) == (16, 16),   "stride 2"
assert conv2d_output_shape(224, 224, 7, P=3, S=2) == (112, 112), "ResNet conv1"
assert conv2d_output_shape(7, 7, 3, P=0, S=1) == (5, 5)
print("Part 1 passed!")

With the output-shape formula in hand, let us implement the convolution itself.

## Part 2 (25 points, coding task)

**Do the following tasks.**

Implement `my_conv2d(x, weight, bias, stride, padding)` using `F.unfold` (im2col) to convert the convolution into a matrix multiplication.

- `x`: $(B, C_{\text{in}}, H, W)$.
- `weight`: $(C_{\text{out}}, C_{\text{in}}, K_h, K_w)$.
- `bias`: $(C_{\text{out}},)$ or `None`.
- Returns: $(B, C_{\text{out}}, H_{\text{out}}, W_{\text{out}})$.

Steps:
1. Use `F.unfold(x, kernel_size, padding=padding, stride=stride)` to extract patches → shape `(B, C_in*K*K, L)` where $L = H_{\text{out}} \times W_{\text{out}}$.
2. Reshape `weight` to `(C_out, C_in*K*K)`.
3. Matrix multiply and reshape to `(B, C_out, H_out, W_out)`.
4. Add bias if not None.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def my_conv2d(x, weight, bias=None, stride=1, padding=0):
    """
    2D convolution via im2col (F.unfold).
    x: (B, C_in, H, W), weight: (C_out, C_in, Kh, Kw)
    Returns: (B, C_out, H_out, W_out)
    """
    ...

""" END OF THIS PART """

In [ ]:
""" VERIFICATION """
torch.manual_seed(42)
x = torch.randn(2, 3, 8, 8)

# Test 1: 3x3 conv with padding
conv = nn.Conv2d(3, 16, 3, padding=1)
out_mine = my_conv2d(x, conv.weight, conv.bias, stride=1, padding=1)
out_ref = conv(x)
assert torch.allclose(out_mine, out_ref, atol=1e-5), f"Max error: {(out_mine - out_ref).abs().max():.6f}"

# Test 2: stride 2
conv2 = nn.Conv2d(3, 8, 3, stride=2, padding=1)
out_mine2 = my_conv2d(x, conv2.weight, conv2.bias, stride=2, padding=1)
out_ref2 = conv2(x)
assert out_mine2.shape == out_ref2.shape
assert torch.allclose(out_mine2, out_ref2, atol=1e-5)

# Test 3: no bias
conv3 = nn.Conv2d(3, 4, 5, bias=False, padding=0)
out_mine3 = my_conv2d(x, conv3.weight, None, stride=1, padding=0)
out_ref3 = conv3(x)
assert torch.allclose(out_mine3, out_ref3, atol=1e-5)

print("Part 2 passed!")

Pooling layers reduce spatial dimensions. Let us implement max pooling.

## Part 3 (15 points, coding task)

**Do the following tasks.**

Implement `my_maxpool2d(x, kernel_size, stride)` without `nn.MaxPool2d` or `F.max_pool2d`.

- `x`: $(B, C, H, W)$.
- `kernel_size`: integer (e.g., 2).
- `stride`: integer (defaults to `kernel_size` if `None`).
- Returns: $(B, C, H_{\text{out}}, W_{\text{out}})$.

Hint: use `x.unfold(dim, size, step)` twice (once for height, once for width), then take the max.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def my_maxpool2d(x, kernel_size=2, stride=None):
    """
    Max pooling 2D from scratch.
    x: (B, C, H, W) -> (B, C, H_out, W_out)
    """
    if stride is None:
        stride = kernel_size
    ...

""" END OF THIS PART """

In [ ]:
""" VERIFICATION """
torch.manual_seed(42)
x = torch.randn(2, 3, 8, 8)

pool = nn.MaxPool2d(2, 2)
out_mine = my_maxpool2d(x, 2, 2)
out_ref = pool(x)
assert out_mine.shape == out_ref.shape, f"Shape: {out_mine.shape} vs {out_ref.shape}"
assert torch.allclose(out_mine, out_ref)

# Different kernel
x2 = torch.randn(4, 1, 6, 6)
pool2 = nn.MaxPool2d(3, 3)
assert torch.allclose(my_maxpool2d(x2, 3, 3), pool2(x2))

print("Part 3 passed!")

Let us assemble a full CNN using our from-scratch building blocks.

## Part 4 (20 points, coding task)

**Do the following tasks.**

Build a CNN for CIFAR-10 (input $3 \times 32 \times 32$) using `my_conv2d` and `my_maxpool2d`. Store convolution weights as `nn.Parameter` objects (no `nn.Conv2d`).

Architecture:
```
my_conv2d(3, 16, K=3, pad=1) -> ReLU -> my_maxpool2d(2) ->
my_conv2d(16, 32, K=3, pad=1) -> ReLU -> my_maxpool2d(2) ->
Flatten -> Linear(32*8*8, 10)
```

Initialize convolution weights with `nn.init.kaiming_uniform_`.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class MyScratchCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        ...
    
    def forward(self, x):
        """x: (B, 3, 32, 32) -> (B, 10)"""
        ...

""" END OF THIS PART """

In [ ]:
""" VERIFICATION """
torch.manual_seed(42)
model = MyScratchCNN()
x = torch.randn(4, 3, 32, 32)  # batch of 4 CIFAR-10 images
logits = model(x)
assert logits.shape == (4, 10), f"Expected (4, 10), got {logits.shape}"
total_params = sum(p.numel() for p in model.parameters())
assert total_params > 0
print(f"Part 4 passed! Total parameters: {total_params:,}")

Understanding the im2col approach and parameter efficiency is critical for contest problems.

## Part 5 (10 points, non-coding task)

**Do the following tasks (Reasoning is required).**

1. In the im2col approach, we reshape the input into a matrix $X_{\text{col}}$ and the kernels into a matrix $W_{\text{col}}$, then compute $Y = W_{\text{col}} \cdot X_{\text{col}}$. For input $(B, C_{\text{in}}, H, W)$ with kernel size $K$ and stride $S$, what are the shapes of $X_{\text{col}}$, $W_{\text{col}}$, and $Y$?

2. A standard `Conv2d(64, 128, 3)` (with bias) has how many parameters? A `Conv2d(64, 128, 1)` (with bias)? What does the $1 \times 1$ convolution do conceptually (no spatial mixing)?

3. **Global Average Pooling** maps $(B, C, H, W) \to (B, C)$ by averaging over $H \times W$. How many parameters does it have? Why is it preferred over `Flatten -> Linear` at the end of a CNN?

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

To solidify understanding, let us trace the shapes and parameter counts through a realistic CNN layer by layer.

## Part 6 (10 points, coding task)

**Do the following tasks.**

Trace every intermediate shape through the following VGG-style network on a single input of shape `(1, 3, 32, 32)`. You may use `nn.Conv2d` here.

```python
vgg_mini = nn.Sequential(
    nn.Conv2d(3, 64, 3, padding=1), nn.ReLU(),      # block 1
    nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(),
    nn.MaxPool2d(2, 2),
    nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),     # block 2
    nn.Conv2d(128, 128, 3, padding=1), nn.ReLU(),
    nn.MaxPool2d(2, 2),
    nn.AdaptiveAvgPool2d(1),                          # GAP
    nn.Flatten(),
    nn.Linear(128, 10),
)
```

Store a list `shapes` where each element is a tuple `(layer_name, output_shape_tuple)`. Use forward hooks or manual tracing.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

shapes = []  # list of (str, tuple) pairs

...

""" END OF THIS PART """

In [ ]:
""" VERIFICATION """
assert len(shapes) >= 5, f"Expected at least 5 shape entries, got {len(shapes)}"
assert shapes[-1][1][-1] == 10, "Final output should have 10 classes"
print("Part 6 passed!")
for name, shape in shapes:
    print(f"  {name}: {shape}")

## Part 7 (10 points, non-coding task)

**Do the following tasks (Reasoning is required).**

**Receptive field analysis.**

For a stack of $L$ convolutional layers, each with kernel size $K$ and stride $S$, the receptive field of the output is:

$$r_L = 1 + \sum_{l=1}^{L}(K_l - 1)\prod_{i=1}^{l-1} S_i$$

1. For the VGG-mini network in Part 6 (four `Conv2d(3, pad=1)` layers and two `MaxPool2d(2, 2)` layers interleaved), compute the receptive field of the final feature map (before Global Average Pooling).

2. Two stacked $3 \times 3$ convolutions have the same receptive field as a single $5 \times 5$ convolution. Compare the parameter counts (assume $C$ input and $C$ output channels, with bias). Which is more efficient and by what factor?

3. ResNet uses $7 \times 7$ convolution with stride 2 as its first layer on $224 \times 224$ input. What is the output spatial size? Why start with a large kernel?

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """